<a target="_blank" href="https://colab.research.google.com/github/tuliosg/linguistica-computacional/blob/main/notebooks/3-Introdu%C3%A7%C3%A3o%20ao%20PLN.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# **Introdução ao Processamento de Linguagem Natural**
Vamos conhecer duas das bibliotecas mais utilizadas para PLN: spaCy e NLTK.

### **Dependências**

In [1]:
#liste todas as dependências para instalação
%pip install spacy nltk pandas openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import nltk
import pandas as pd
import spacy
from nltk.tokenize import word_tokenize

spacy.cli.download("pt_core_news_sm")
nltk.download('punkt')
nltk.download('punkt_tab')

✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tulio\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
!mkdir -p ../content/data

!curl -L -o ../content/data/noticia_sintetica.txt https://raw.githubusercontent.com/tuliosg/linguistica-computacional/refs/heads/main/data/noticia_sintetica.txt
!curl -L -o ../content/data/ent_sintetica.xlsx https://raw.githubusercontent.com/tuliosg/linguistica-computacional/refs/heads/main/data/ent_sintetica.xlsx

### **Carregando os dados**

In [ ]:
conteudo_txt = open('./data/noticia_sintetica.txt', 'r', encoding='utf-8').read()
eaf = pd.read_excel('./data/ent_sintetica.xlsx')

## **Tokenização**
A tokenização consiste em dividir um texto em unidades menores chamadas **tokens**
Na maioria das aplicações essas unidades correspondem às palavras, embora também possam representar pontuação, números ou até mesmo partes de uma palavra (como nos LLMs).
Essa é uma das etapas fundamentais do pré-processamento textual, pois praticamente todas as tarefas de PLN trabalham sobre tokens.

Vamos ver três maneiras diferentes de realizar essa tarefa.

### **`split`**

In [5]:
tokens_split = conteudo_txt.split()

display(tokens_split[:30])

['Centro',
 'de',
 'pesquisa',
 'lança',
 'plataforma',
 'aberta',
 'para',
 'organização',
 'de',
 'dados',
 'linguísticos',
 'Na',
 'última',
 'sexta-feira,',
 'pesquisadores',
 'do',
 'Centro',
 'de',
 'Estudos',
 'em',
 'Linguagem',
 'e',
 'Tecnologia',
 '(CELT)',
 'apresentaram',
 'uma',
 'nova',
 'plataforma',
 'voltada',
 'para']

In [6]:
print(len(tokens_split))

457


Embora extremamente simples, essa abordagem apenas separa o texto pelos espaços em branco.
Como consequência, a pontuação permanece "presa" às palavras.

### **Natural Language Toolkit**
O NLTK implementa tokenizadores desenvolvidos especificamente para processamento textual.


In [7]:
tokens_nltk = word_tokenize(
    conteudo_txt,
    language="portuguese"
)
display(tokens_nltk[:30])

['Centro',
 'de',
 'pesquisa',
 'lança',
 'plataforma',
 'aberta',
 'para',
 'organização',
 'de',
 'dados',
 'linguísticos',
 'Na',
 'última',
 'sexta-feira',
 ',',
 'pesquisadores',
 'do',
 'Centro',
 'de',
 'Estudos',
 'em',
 'Linguagem',
 'e',
 'Tecnologia',
 '(',
 'CELT',
 ')',
 'apresentaram',
 'uma',
 'nova']

In [8]:
print(len(tokens_nltk))

505


Observem que agora a pontuação passa a ser tratada como um token independente.

### **spaCy**
O spaCy é uma biblioteca completa de PLN que traz desde técnicas básicas até avançadas para trabalharmos com texto.

In [9]:
nlp = spacy.load("pt_core_news_sm")
doc = nlp(conteudo_txt)

tokens_spacy = [token.text for token in doc]

tokens_spacy[:30]

['Centro',
 'de',
 'pesquisa',
 'lança',
 'plataforma',
 'aberta',
 'para',
 'organização',
 'de',
 'dados',
 'linguísticos',
 '\n\n',
 'Na',
 'última',
 'sexta-feira',
 ',',
 'pesquisadores',
 'do',
 'Centro',
 'de',
 'Estudos',
 'em',
 'Linguagem',
 'e',
 'Tecnologia',
 '(',
 'CELT',
 ')',
 'apresentaram',
 'uma']

Vamos comparar os resultados:

In [10]:
print("split():", len(tokens_split))
print("NLTK:", len(tokens_nltk))
print("spaCy:", len(tokens_spacy))

split(): 457
NLTK: 505
spaCy: 513



Embora semelhantes, cada biblioteca utiliza regras próprias para identificar limites entre tokens. Dentre elas, a mais interessante é a do spaCy, pois separa todos os tipos de caracteres.

A escolha da ferramenta depende dos objetivos da pesquisa.

### **Explorando os tokens**

In [11]:
doc = nlp(conteudo_txt)

tokens = list(doc[:10])

df_tokens = pd.DataFrame(
    [
        {
            "Token": token.text,
            "Classe gramatical": token.pos_,
            "Lema": token.lemma_,
            "Radical": token.norm_,
            "Dependência": token.dep_
        }
        for token in tokens
    ]
)

df_tokens

,Token,Classe gramatical,Lema,Radical,Dependência
0,Centro,NOUN,centro,centro,nsubj
1,de,ADP,de,de,case
2,pesquisa,NOUN,pesquisa,pesquisa,nmod
3,lança,VERB,lançar,lança,ROOT
4,plataforma,NOUN,plataforma,plataforma,obj
5,aberta,VERB,abrir,aberta,acl
6,para,ADP,para,para,case
7,organização,NOUN,organização,organização,obj
8,de,ADP,de,de,case
9,dados,NOUN,dado,dados,nmod


Observe que cada token deixa de ser apenas uma palavra.

O spaCy associa automaticamente diversas informações linguísticas a cada elemento do texto, permitindo análises muito mais sofisticadas.

Neste exemplo destacamos:

- **Classe gramatical (POS):** categoria sintática do token (substantivo, verbo, adjetivo etc.);
- **Lema:** forma canônica da palavra;
- **Radical (forma normalizada):** representação simplificada utilizada pelo modelo;
- **Dependência sintática:** relação entre o token e os demais elementos da sentença.

## **Busca de contextos avançada**
Vamos localizar todas as ocorrências da palavra **dados**.
Além da palavra, queremos visualizar as cinco palavras anteriores e posteriores.

In [12]:
def buscar_contextos(tokens, palavra, janela=5):
    """
    Busca todas as ocorrências de uma palavra em uma sequência de tokens e
    recupera seu contexto de ocorrência.

    Para cada ocorrência encontrada, a função retorna a posição da palavra na
    lista de tokens, a palavra localizada e uma janela de contexto contendo um
    número configurável de tokens anteriores e posteriores.

    Parameters
    ----------
    tokens : list[str]
        Lista de tokens que será utilizada na busca.

    palavra : str
        Palavra que será procurada.

    janela : int, optional
        Número de tokens que devem ser recuperados antes e depois da palavra
        encontrada. O padrão é 5.

    Returns
    -------
    list[dict]
        Lista de dicionários contendo:

        - indice : posição da palavra na lista de tokens;
        - palavra : palavra encontrada;
        - contexto : trecho de texto contendo a palavra e sua vizinhança.
    """

    resultados = []

    for indice, token in enumerate(tokens):

        if token.lower() == palavra.lower():

            inicio = max(0, indice - janela)
            fim = indice + janela + 1

            contexto = tokens[inicio:fim]

            resultados.append(
                {
                    "indice": indice,
                    "palavra": token,
                    "contexto": " ".join(contexto)
                }
            )

    return resultados

In [13]:
contextos = buscar_contextos(
    tokens_spacy,
    palavra="dados",
    janela=5
)

contextos[:5]

[{'indice': 9,
  'palavra': 'dados',
  'contexto': 'plataforma aberta para organização de dados linguísticos \n\n Na última sexta-feira'},
 {'indice': 41,
  'palavra': 'dados',
  'contexto': ', documentação e compartilhamento de dados linguísticos produzidos em projetos de'},
 {'indice': 80,
  'palavra': 'dados',
  'contexto': 'recomendações internacionais para gestão de dados científicos . \n\n Segundo a'},
 {'indice': 98,
  'palavra': 'dados',
  'contexto': 'é tornar os conjuntos de dados mais fáceis de encontrar ,'},
 {'indice': 136,
  'palavra': 'dados',
  'contexto': 'sempre que a divulgação dos dados for eticamente possível . \n\n'}]

## **Transformando os resultados em uma planilha**

In [14]:
df_contextos = pd.DataFrame(contextos)

df_contextos

,indice,palavra,contexto
0,9,dados,plataforma aberta para organização de dados li...
1,41,dados,", documentação e compartilhamento de dados lin..."
2,80,dados,recomendações internacionais para gestão de da...
3,98,dados,é tornar os conjuntos de dados mais fáceis de ...
4,136,dados,sempre que a divulgação dos dados for eticamen...
5,194,dados,processamento e a exportação dos dados em dife...
6,215,dados,que a gestão adequada dos dados reduz o risco ...
7,250,dados,consistente torna os conjuntos de dados mais ú...
8,322,dados,tempo necessário para preparar os dados antes ...
9,400,dados,incentivar o compartilhamento responsável dos ...


In [ ]:
df_contextos.to_csv(
    "./data/contextos_noticia.xlsx",
    index=False
)

## **Etiquetagem morfossintática**
A etiquetagem morfossintática (Part-of-Speech Tagging ou POS Tagging) consiste em atribuir uma classe gramatical a cada token de um texto. Essas etiquetas permitem identificar substantivos, verbos, adjetivos, pronomes e outras categorias, sendo amplamente utilizadas em tarefas de análise linguística e Processamento de Linguagem Natural.

### **spaCy**

In [16]:
doc = nlp(conteudo_txt)

# Organizando os resultados em uma tabela
df_spacy = pd.DataFrame(
    [
        {
            "Token": token.text,
            "Lema": token.lemma_,
            "Classe": token.pos_,
            "Descrição": spacy.explain(token.pos_),
            "Contexto": " ".join(
                t.text
                for t in doc[max(0, token.i - 3): token.i + 4]
            )
        }
        for token in doc
    ]
)

df_spacy.head(30)

,Token,Lema,Classe,Descrição,Contexto
0,Centro,centro,NOUN,noun,Centro de pesquisa lança
1,de,de,ADP,adposition,Centro de pesquisa lança plataforma
2,pesquisa,pesquisa,NOUN,noun,Centro de pesquisa lança plataforma aberta
3,lança,lançar,VERB,verb,Centro de pesquisa lança plataforma aberta para
4,plataforma,plataforma,NOUN,noun,de pesquisa lança plataforma aberta para organ...
5,aberta,abrir,VERB,verb,pesquisa lança plataforma aberta para organiza...
6,para,para,ADP,adposition,lança plataforma aberta para organização de dados
7,organização,organização,NOUN,noun,plataforma aberta para organização de dados li...
8,de,de,ADP,adposition,aberta para organização de dados linguísticos ...
9,dados,dado,NOUN,noun,para organização de dados linguísticos \n\n Na


<div style="text-align: right; background-color: #f8f9fa; padding: 10px 15px; border-right: 4px solid #404040; border-radius: 5px; margin-top: 15px; display: inline-block; float: right; min-width: 300px;">
    <span style="font-weight: bold; color: #404040; font-size: 0.95em;">Métodos computacionais para linguística</span><br>
    <span style="color: #6c757d; font-size: 0.85em; font-style: italic;">Explorando fenômenos linguísticos</span><br>
    <span style="color: #595959; font-size: 0.8em; margin-top: 5px; display: inline-block;">Desenvolvido por <strong><a href="https://github.com/tuliosg" target="_blank" style="color: #404040; text-decoration: none;">Túlio Gois</a></strong></span>
</div>
<div style="clear: both;"></div>